# FinHybrid - Phase 3B: FinBERT Embeddings

**Optimization:** Phase 3B - FinBERT domain-specific embeddings

**Dataset:** FinHybrid (Financial Reports)

**Model:** NVIDIA Nemotron-3 Ultra 550B (via Together AI)

**Metric:** Exact Match ±1%

**Documents:** 4 example PDFs (ADI, ABMD, GS, JKHY)

**What changed:**
- ✅ FinBERT embeddings (financial domain-specific)
- ✅ Chain-of-thought prompt (from Phase 3C best)
- ✅ All Phase 2 parameters (TOP_K=10, CHUNK_SIZE=1500)

**Baseline (Phase 3C with generic embeddings):**
- Empty rate: 27.7% (13/47 questions)
- Exact Match: 6.38%

**Expected with FinBERT:**
- Empty rate: 23-26% (+2-4 questions)
- Better understanding of financial terminology
- Better retrieval of numerical contexts

**Expected runtime:** 30-40 minutes (slower embeddings)
**Expected cost:** $0 additional (free embeddings, same LLM calls)

## Setup and Imports

In [1]:
import sys
import os

# Navigate to project root
project_root = os.path.abspath('../../../../..')
os.chdir(project_root)
sys.path.insert(0, project_root)

print(f"Working directory: {os.getcwd()}")

Working directory: /Users/I772947/personal work/LLM Benchmark Team Project/LLM_Benchmark_Team_Project_2026/Sprint 3/UDA-Benchmark


In [2]:
import pandas as pd
import chromadb
import PyPDF2
import time
import importlib.util
from datetime import datetime
from together import Together
from langchain.text_splitter import RecursiveCharacterTextSplitter
from uda.utils import preprocess
from uda.utils.prompts import get_prompt
from uda.utils.embeddings import get_finbert_embedding_function  # NEW: FinBERT embeddings
from uda.eval.my_eval import eval_main

print("✓ All imports successful")

/Users/I772947/Library/Python/3.9/lib/python/site-packages/urllib3/__init__.py:35: NotOpenSSLWarning: urllib3 v2 only supports OpenSSL 1.1.1+, currently the 'ssl' module is compiled with 'LibreSSL 2.8.3'. See: https://github.com/urllib3/urllib3/issues/3020
  warnings.warn(
/Users/I772947/Library/Python/3.9/lib/python/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


✓ All imports successful


## Configuration

In [3]:
# Load API config
_spec = importlib.util.spec_from_file_location(
    "access_config",
    os.path.join(os.getcwd(), "uda", "utils", "access_config.py")
)
access_config = importlib.util.module_from_spec(_spec)
_spec.loader.exec_module(access_config)

print(f"Model: {access_config.TOGETHER_MODEL}")
print(f"API Key: {access_config.TOGETHER_API_KEY[:20]}...")

Model: nvidia/nemotron-3-ultra-550b-a55b
API Key: tgp_v1_9OcdTuqoXTB0_...


In [4]:
# Experiment Parameters (Phase 3C best + FinBERT)
DATASET_NAME = "fin"
CHUNK_SIZE = 1500  # From Phase 2
CHUNK_OVERLAP = 150
TOP_K = 10  # From Phase 2
TEMPERATURE = 0.1
MAX_TOKENS = 512

# Prompt type (best from Phase 3C)
PROMPT_TYPE = "cot"  # Chain-of-thought won for FinHybrid

# NEW: Embedding type
EMBEDDING_TYPE = "finbert"  # financial domain-specific

# Output settings
TIMESTAMP = datetime.now().strftime("%Y%m%d_%H%M%S")
OUTPUT_DIR = "./experiments/nemotron-3-ultra-550b/3_advanced_optimization/2_finbert/results/finhybrid_finbert"
os.makedirs(OUTPUT_DIR, exist_ok=True)

print(f"Dataset: {DATASET_NAME}")
print(f"Chunk size: {CHUNK_SIZE}")
print(f"Top-K: {TOP_K}")
print(f"Prompt type: {PROMPT_TYPE}")
print(f"Embedding type: {EMBEDDING_TYPE}")
print(f"Output dir: {OUTPUT_DIR}")

Dataset: fin
Chunk size: 1500
Top-K: 10
Prompt type: cot
Embedding type: finbert
Output dir: ./experiments/nemotron-3-ultra-550b/3_advanced_optimization/2_finbert/results/finhybrid_finbert


## Initialize Models

In [5]:
# Together AI client
together_client = Together(api_key=access_config.TOGETHER_API_KEY)
print("✓ Together AI client initialized")

# NEW: FinBERT embedding model (financial domain-specific)
print("Loading FinBERT embeddings...")
ef = get_finbert_embedding_function()
print("✓ FinBERT embedding model loaded")
print("  → Better understanding of financial terminology")
print("  → Better semantic matching for numerical contexts")

# Text splitter
text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=CHUNK_SIZE,
    chunk_overlap=CHUNK_OVERLAP,
)
print("✓ Text splitter initialized")

# Prompt function
prompt_fn = get_prompt(PROMPT_TYPE)
print(f"✓ Prompt function loaded: {PROMPT_TYPE}")

INFO:uda.utils.embeddings:Loading FinBERT model: yiyanghkust/finbert-tone
INFO:sentence_transformers.SentenceTransformer:Use pytorch device_name: mps
INFO:sentence_transformers.SentenceTransformer:Load pretrained SentenceTransformer: yiyanghkust/finbert-tone


✓ Together AI client initialized
Loading FinBERT embeddings...


INFO:uda.utils.embeddings:✅ FinBERT loaded. Embedding dimension: 768


✓ FinBERT embedding model loaded
  → Better understanding of financial terminology
  → Better semantic matching for numerical contexts
✓ Text splitter initialized
✓ Prompt function loaded: cot


## Helper Functions

In [6]:
import re

def extract_pdf_text(pdf_path):
    """Extract text from PDF using PyPDF2"""
    pdf_text = ""
    with open(pdf_path, "rb") as file:
        reader = PyPDF2.PdfReader(file, strict=False)
        for page_num in range(len(reader.pages)):
            pdf_text += reader.pages[page_num].extract_text()
    return pdf_text

def sanitize_collection_name(doc_name, dataset_name):
    """ChromaDB requires [a-zA-Z0-9._-], start/end with alphanumeric"""
    safe_name = re.sub(r'[^a-zA-Z0-9._-]', '_', doc_name)
    safe_name = re.sub(r'_+', '_', safe_name)
    safe_name = safe_name.strip('_')
    return f"{dataset_name}_{safe_name}"

def build_index(text_chunks, doc_name):
    """Build vector index with FinBERT embeddings"""
    chroma_client = chromadb.Client()
    
    # Create safe collection name
    collection_name = sanitize_collection_name(doc_name, DATASET_NAME)
    
    # Delete if exists
    try:
        chroma_client.delete_collection(collection_name)
    except:
        pass
    
    # Create collection with FinBERT embeddings
    collection = chroma_client.create_collection(
        collection_name,
        embedding_function=ef,  # FinBERT embedding function
        metadata={"hnsw:space": "cosine"}
    )
    
    # Add documents
    id_list = [str(i) for i in range(len(text_chunks))]
    collection.add(documents=text_chunks, ids=id_list)
    
    return collection

def answer_question(collection, question):
    """
    Retrieve context and generate answer.
    
    Uses FinBERT for retrieval (better financial semantic matching)
    and chain-of-thought prompt for generation.
    """
    # Retrieve (using FinBERT embeddings)
    fetch_res = collection.query(query_texts=[question], n_results=TOP_K)
    context = "\n".join(fetch_res["documents"][0])
    
    # Build prompt
    prompt_text = prompt_fn(context=context, question=question)
    
    # Convert to message format
    messages = [
        {"role": "user", "content": prompt_text}
    ]
    
    # Generate
    response = together_client.chat.completions.create(
        model=access_config.TOGETHER_MODEL,
        messages=messages,
        temperature=TEMPERATURE,
        max_tokens=MAX_TOKENS,
    )
    
    return response.choices[0].message.content

print("✓ Helper functions defined")

✓ Helper functions defined


## Load Q&A Data

In [7]:
# Load FinHybrid Q&A
csv_file = "./dataset/qa/fin_qa.csv"
df = pd.read_csv(csv_file, sep="|", na_filter=False, dtype={"doc_name": str})
qas_dict_all = preprocess.qa_df_to_dict(DATASET_NAME, df)

# Same documents as Phase 3C (for valid comparison)
AVAILABLE_DOCS = [
    "ABMD_2012",
    "ADI_2009",
    "GS_2016",
    "JKHY_2015"
]

qas_dict = {doc: qas for doc, qas in qas_dict_all.items() if doc in AVAILABLE_DOCS}

print(f"Total documents in CSV: {len(qas_dict_all)}")
print(f"Available PDFs: {len(AVAILABLE_DOCS)}")
print(f"\nFiltered to documents with PDFs:\n")

total_qa = 0
for doc in AVAILABLE_DOCS:
    if doc in qas_dict:
        count = len(qas_dict[doc])
        total_qa += count
        print(f"  {doc}: {count} Q&A pairs")

print(f"\nTotal Q&A to process: {total_qa}")

Total documents in CSV: 788
Available PDFs: 4

Filtered to documents with PDFs:

  ABMD_2012: 12 Q&A pairs
  ADI_2009: 9 Q&A pairs
  GS_2016: 23 Q&A pairs
  JKHY_2015: 3 Q&A pairs

Total Q&A to process: 47


## Main Processing Loop

**This will process 4 documents with 47 Q&A pairs**

**Expected runtime:** 30-40 minutes (FinBERT embedding generation is slower)

In [8]:
all_results = []

for doc_name, doc_qas in qas_dict.items():
    print(f"\n{'='*80}")
    print(f"Processing: {doc_name}")
    print(f"{'='*80}")
    
    # Get PDF path
    pdf_path = preprocess.get_example_pdf_path(DATASET_NAME, doc_name)
    if not pdf_path:
        print(f"❌ PDF not found - skipping")
        continue
    
    print(f"PDF: {pdf_path}")
    
    # Extract and chunk
    print("Extracting text...")
    pdf_text = extract_pdf_text(pdf_path)
    text_chunks = text_splitter.split_text(pdf_text)
    print(f"Created {len(text_chunks)} chunks")
    
    # Build index (FinBERT embeddings - slower but better)
    print("Building vector index with FinBERT embeddings...")
    print("  (This will take longer than generic embeddings)")
    collection = build_index(text_chunks, doc_name)
    print("✓ Index built with financial domain understanding")
    
    # Process each question
    print(f"\nAnswering {len(doc_qas)} questions...")
    
    for idx, qa in enumerate(doc_qas, 1):
        question = qa["question"]
        print(f"\n[{idx}/{len(doc_qas)}] {question[:70]}...")
        
        try:
            answer = answer_question(collection, question)
            print(f"   Answer: {answer[:80]}...")
            
            all_results.append({
                "question": question,
                "response": answer,
                "doc": doc_name,
                "q_uid": qa["q_uid"],
                "answers": qa["answers"],
                "dataset": DATASET_NAME,
                "prompt_type": PROMPT_TYPE,
                "embedding_type": EMBEDDING_TYPE,
            })
            
            time.sleep(0.5)  # Rate limiting
            
        except Exception as e:
            print(f"   ❌ Error: {e}")
            continue
    
    print(f"\n✓ Completed {doc_name}: {len([r for r in all_results if r['doc'] == doc_name])} questions processed")

print(f"\n{'='*80}")
print(f"ALL DOCUMENTS PROCESSED")
print(f"{'='*80}")
print(f"Total Q&A processed: {len(all_results)}")


Processing: ADI_2009
PDF: dataset/src_doc_files_example/fin_docs/ADI_2009.pdf
Extracting text...
Created 284 chunks
Building vector index with FinBERT embeddings...
  (This will take longer than generic embeddings)


Batches: 100%|██████████| 9/9 [00:13<00:00,  1.50s/it]


✓ Index built with financial domain understanding

Answering 9 questions...

[1/9] what is the the interest expense in 2009?...


Batches: 100%|██████████| 1/1 [00:00<00:00,  2.74it/s]
INFO:httpx:HTTP Request: POST https://api.together.xyz/v1/chat/completions "HTTP/1.1 200 OK"


   Answer: Based on the provided context, the **total interest expense for fiscal 2009 is n...

[2/9] what is the expected growth rate in amortization expense in 2010?...


...

   Answer: Based on the performance graph data in the context, the Peer Group index values ...

[2/3] in 2010 , what was the cumulative total return of the s&p 500?...


Batches: 100%|██████████| 1/1 [00:00<00:00,  8.50it/s]
INFO:httpx:HTTP Request: POST https://api.together.xyz/v1/chat/completions "HTTP/1.1 200 OK"


   Answer: Based on the performance graph in the 2015 annual report, the cumulative total r...

[3/3] for the 2010 , what was the cumulative total return on jkhy?...


Batches: 100%|██████████| 1/1 [00:00<00:00,  9.51it/s]
INFO:httpx:HTTP Request: POST https://api.together.xyz/v1/chat/completions "HTTP/1.1 200 OK"


   Answer: Based on the Performance Graph in the annual report, the cumulative total return...

✓ Completed JKHY_2015: 3 questions processed

ALL DOCUMENTS PROCESSED
Total Q&A processed: 47


## Diagnostic: Check Empty Responses

In [9]:
if all_results:
    results_df = pd.DataFrame(all_results)
    
    # Count empty responses
    results_df['is_empty'] = results_df['response'].fillna('').str.strip() == ''
    empty_count = results_df['is_empty'].sum()
    total_count = len(results_df)
    
    print(f"\n{'='*80}")
    print(f"DIAGNOSTIC: Empty Response Analysis")
    print(f"{'='*80}")
    print(f"Total Q&A processed: {total_count}")
    print(f"Empty responses: {empty_count} ({empty_count/total_count*100:.1f}%)")
    print(f"Answered: {total_count - empty_count} ({(total_count-empty_count)/total_count*100:.1f}%)")
    
    # COMPARISON WITH PHASE 3C (generic embeddings)
    phase3c_empty = 13
    phase3c_total = 47
    phase3c_empty_pct = phase3c_empty / phase3c_total * 100
    
    improvement = phase3c_empty - empty_count
    improvement_pct = phase3c_empty_pct - (empty_count/total_count*100)
    
    print(f"\n{'='*80}")
    print(f"COMPARISON WITH PHASE 3C (Generic Embeddings)")
    print(f"{'='*80}")
    print(f"Phase 3C (Generic): {phase3c_empty}/{phase3c_total} empty ({phase3c_empty_pct:.1f}%)")
    print(f"Phase 3B (FinBERT): {empty_count}/{total_count} empty ({empty_count/total_count*100:.1f}%)")
    print(f"\nImprovement: {improvement:+d} questions ({improvement_pct:+.1f} percentage points)")
    
    if improvement >= 2:
        print(f"✅ SUCCESS: FinBERT embeddings improved financial retrieval!")
    elif improvement > 0:
        print(f"⚠️  WEAK: Marginal improvement")
    elif improvement == 0:
        print(f"⚠️  NEUTRAL: No change")
    else:
        print(f"❌ REGRESSION: Generic embeddings were better")
    
    # Phase 2 comparison for reference
    phase2_empty = 17
    total_improvement = phase2_empty - empty_count
    
    print(f"\n{'='*80}")
    print(f"TOTAL IMPROVEMENT FROM PHASE 2 BASELINE")
    print(f"{'='*80}")
    print(f"Phase 2 (Baseline): {phase2_empty}/47 empty (36.2%)")
    print(f"Phase 3B (FinBERT): {empty_count}/47 empty ({empty_count/47*100:.1f}%)")
    print(f"Total improvement: {total_improvement:+d} questions")
    
    if empty_count > 0:
        print(f"\nEmpty responses by document:")
        for doc in results_df['doc'].unique():
            doc_df = results_df[results_df['doc'] == doc]
            doc_empty = doc_df['is_empty'].sum()
            doc_total = len(doc_df)
            print(f"  {doc}: {doc_empty}/{doc_total} empty ({doc_empty/doc_total*100:.1f}%)")
else:
    print("❌ No results to analyze")


DIAGNOSTIC: Empty Response Analysis
Total Q&A processed: 47
Empty responses: 20 (42.6%)
Answered: 27 (57.4%)

COMPARISON WITH PHASE 3C (Generic Embeddings)
Phase 3C (Generic): 13/47 empty (27.7%)
Phase 3B (FinBERT): 20/47 empty (42.6%)

Improvement: -7 questions (-14.9 percentage points)
❌ REGRESSION: Generic embeddings were better

TOTAL IMPROVEMENT FROM PHASE 2 BASELINE
Phase 2 (Baseline): 17/47 empty (36.2%)
Phase 3B (FinBERT): 20/47 empty (42.6%)
Total improvement: -3 questions

Empty responses by document:
  ADI_2009: 6/9 empty (66.7%)
  ABMD_2012: 3/12 empty (25.0%)
  GS_2016: 11/23 empty (47.8%)
  JKHY_2015: 0/3 empty (0.0%)


## Evaluate Results

In [10]:
if all_results:
    print("\nEvaluating FinHybrid results (Exact Match ±1%)...")
    eval_main(DATASET_NAME, all_results)
else:
    print("❌ No results to evaluate")


Evaluating FinHybrid results (Exact Match ±1%)...
Exact-match accuracy: 2.13


## Save Results

In [11]:
if all_results:
    results_df = pd.DataFrame(all_results)
    output_file = os.path.join(OUTPUT_DIR, f"finhybrid_finbert_{TIMESTAMP}.csv")
    results_df.to_csv(output_file, index=False)
    
    print(f"\n✓ Results saved to: {output_file}")
    print(f"Total Q&A: {len(results_df)}")
    
    # Summary by document
    print("\nResults by document:")
    for doc in results_df['doc'].unique():
        count = len(results_df[results_df['doc'] == doc])
        print(f"  {doc}: {count} questions")
else:
    print("❌ No results to save")


✓ Results saved to: ./experiments/nemotron-3-ultra-550b/3_advanced_optimization/2_finbert/results/finhybrid_finbert/finhybrid_finbert_20260630_140755.csv
Total Q&A: 47

Results by document:
  ADI_2009: 9 questions
  ABMD_2012: 12 questions
  GS_2016: 23 questions
  JKHY_2015: 3 questions


## Final Summary

In [12]:
if all_results:
    results_df = pd.DataFrame(all_results)
    
    empty_count = results_df['response'].fillna('').str.strip().eq('').sum()
    answered_count = len(results_df) - empty_count
    
    phase3c_empty = 13
    improvement = phase3c_empty - empty_count
    
    phase2_empty = 17
    total_improvement = phase2_empty - empty_count
    
    print(f"\n{'='*80}")
    print(f"FINAL SUMMARY - FINBERT EMBEDDINGS")
    print(f"{'='*80}")
    print(f"Dataset: FinHybrid (47 Q&A)")
    print(f"Embedding type: {EMBEDDING_TYPE}")
    print(f"Prompt type: {PROMPT_TYPE}")
    print(f"\nResults:")
    print(f"  Answered: {answered_count}/{len(results_df)} ({answered_count/len(results_df)*100:.1f}%)")
    print(f"  Empty: {empty_count}/{len(results_df)} ({empty_count/len(results_df)*100:.1f}%)")
    print(f"\nVs Phase 3C (Generic embeddings):")
    print(f"  Change: {improvement:+d} questions")
    print(f"  Expected: +2 to +4 questions")
    print(f"  Cost: $0 (free embeddings, same LLM calls)")
    print(f"\nTotal from Phase 2 Baseline:")
    print(f"  Total change: {total_improvement:+d} questions (Phase 2 → Phase 3B)")
    print(f"  Phase 2 empty: 36.2% → Phase 3B: {empty_count/47*100:.1f}%")
    
    if improvement >= 2:
        print(f"\n✅ SUCCESS: FinBERT justified!")
        print(f"   Financial domain understanding improved retrieval")
        print(f"   Next step: Test on TatHybrid (larger financial dataset)")
    elif improvement > 0:
        print(f"\n⚠️  WEAK: Marginal benefit")
        print(f"   FinBERT helps but not dramatically")
        print(f"   Decision: Test TatHybrid before deciding")
    elif improvement == 0:
        print(f"\n⚠️  NEUTRAL: No benefit from FinBERT")
        print(f"   Generic embeddings work just as well")
    else:
        print(f"\n❌ FAILED: Generic embeddings were better")
        print(f"   Recommendation: Stick with generic embeddings")
else:
    print("\n❌ No results to summarize")


FINAL SUMMARY - FINBERT EMBEDDINGS
Dataset: FinHybrid (47 Q&A)
Embedding type: finbert
Prompt type: cot

Results:
  Answered: 27/47 (57.4%)
  Empty: 20/47 (42.6%)

Vs Phase 3C (Generic embeddings):
  Change: -7 questions
  Expected: +2 to +4 questions
  Cost: $0 (free embeddings, same LLM calls)

Total from Phase 2 Baseline:
  Total change: -3 questions (Phase 2 → Phase 3B)
  Phase 2 empty: 36.2% → Phase 3B: 42.6%

❌ FAILED: Generic embeddings were better
   Recommendation: Stick with generic embeddings


---

## Done!

**Results saved to:** `./results/finhybrid_finbert/`

**Next steps:**
1. Analyze FinBERT improvement vs generic embeddings
2. If positive (≥+2 questions), test on TatHybrid
3. Calculate overall Phase 3B impact
4. Determine if <12% overall target achieved